<h1>Data preprocessing

In [ ]:
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, classification_report, 
                             precision_recall_curve, roc_curve, auc,
                             roc_auc_score, average_precision_score) 
from imblearn.over_sampling import SMOTE
import joblib
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
import lightgbm as lgb  

plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = os.path.abspath(os.environ.get("AMD_PAMD_DATA_DIR", os.path.join(os.getcwd(), "data")))
OUTPUT_DIR = os.path.abspath(os.environ.get("AMD_PAMD_OUTPUT_DIR", os.path.join(os.getcwd(), "outputs")))
os.makedirs(OUTPUT_DIR, exist_ok=True)
HEAD_DIR = os.path.join(BASE_DIR, "Sequential structure of the heat map", "head")
TAIL_DIR = os.path.join(BASE_DIR, "Sequential structure of the heat map", "tail")
DATASET_PATH = os.path.join(BASE_DIR, "Dataset.xlsx")

def read_sdf_folder_custom(folder_path):
    """Reads all SDF files in a folder and returns a list of RDKit molecule objects."""
    molecules = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.sdf'):
            filepath = os.path.join(folder_path, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                molblock = f.read()
            mol = Chem.MolFromMolBlock(molblock)
            if mol is not None:
                molecules.append(mol)
    return molecules

all_desc_names = [desc[0] for desc in Descriptors._descList]

def calc_208_descriptors(mol):
    """Calculates 208 RDKit descriptors for a molecule."""
    desc_values = []
    for name in all_desc_names:
        func = getattr(Descriptors, name)
        try:
            val = func(mol)
        except:
            val = 0.0  # Assign 0 if an error occurs
        desc_values.append(val)
    return desc_values

def calc_features(mol):
    """Calculates combined features (Morgan fingerprint + 208 descriptors) for a molecule."""
    # 2048-bit Morgan fingerprint
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    fp_arr = list(fp)
    
    # 208 descriptors
    desc208 = calc_208_descriptors(mol)

    # Concatenate fingerprint and descriptors
    features = fp_arr + desc208
    return features

def feat_name(idx):
    """Returns the name of the feature given its index."""
    if idx < 2048:
        return f"Morgan_{idx}"
    else:
        desc_idx = idx - 2048
        return all_desc_names[desc_idx]


# Paths point to the current workspace data folder
head_mols = read_sdf_folder_custom(HEAD_DIR)
tail_mols = read_sdf_folder_custom(TAIL_DIR)

print(f"The number of head molecules in the training set: {len(head_mols)}")
print(f"The number of tail molecules in the training set: {len(tail_mols)}")

head_features = [calc_features(m) for m in head_mols]
tail_features = [calc_features(m) for m in tail_mols]

head_index = [f"A{i+1}" for i in range(len(head_features))]
tail_index = [f"O{i+1}" for i in range(len(tail_features))]

head_df = pd.DataFrame(head_features, index=head_index)
tail_df = pd.DataFrame(tail_features, index=tail_index)

df = pd.read_excel(DATASET_PATH, index_col=0)  # Rows: head molecule index, Columns: tail molecule index

X = []
y = []

for h in df.index:
    for t in df.columns:
        if str(h) in head_df.index and str(t) in tail_df.index:
            feat = list(head_df.loc[str(h)]) + list(tail_df.loc[str(t)])
            X.append(feat)
            val = df.loc[h, t]
            label = 1 if val >= 2000 else 0  # 1 for good (>= 2000), 0 for poor
            y.append(label)

X = pd.DataFrame(X)
y = pd.Series(y)

print(f"Training sample size: {len(y)}")
print(f"The ratio of positive to negative samples:\n{y.value_counts()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Training set shape after SMOTE: {X_train_res.shape}")
print(f"Target distribution after SMOTE:\n{y_train_res.value_counts()}")

<h1>Train the comparison model

In [ ]:
#################################################################
#                           Model 1: Random Forest
#################################################################
print("\n" + "="*20 + " Model 1: Random Forest " + "="*20 + "\n")

from sklearn.metrics import (
    accuracy_score,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report
)

pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train_res)
X_test_pca = pca.transform(X_test)

param_grid = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [10, 20, 30, 40, None],
    'min_samples_split': [2, 5, 7, 9],
    'class_weight': ['balanced']
}

rf = RandomForestClassifier(random_state=42)

grid = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train_pca, y_train_res)
best_rf_model = grid.best_estimator_

print(f"Optimal model parameters (Random Forest): {grid.best_params_}")

probs_rf = best_rf_model.predict_proba(X_test_pca)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, probs_rf)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-6)

best_idx = np.argmax(f1_scores[:-1])
best_thresh = thresholds[best_idx]

y_pred_rf = (probs_rf >= best_thresh).astype(int)

best_f1 = f1_scores[best_idx]
accuracy = accuracy_score(y_test, y_pred_rf)
macro_f1 = f1_score(y_test, y_pred_rf, average='macro')
weighted_f1 = f1_score(y_test, y_pred_rf, average='weighted')
roc_auc = roc_auc_score(y_test, probs_rf)
pr_auc = average_precision_score(y_test, probs_rf)

print(f"Optimal threshold (Random Forest): {best_thresh:.4f}")
print(f"Best-F1 Score (Random Forest): {best_f1:.4f}")
print(f"Accuracy (Random Forest): {accuracy:.4f}")
print(f"Macro-Avg F1 (Random Forest): {macro_f1:.4f}")
print(f"Weighted-Avg F1 (Random Forest): {weighted_f1:.4f}")
print(f"ROC AUC (Random Forest): {roc_auc:.4f}")
print(f"PR AUC (Random Forest): {pr_auc:.4f}")

print("\nClassification report (Random Forest):")
print(classification_report(y_test, y_pred_rf))

rf_result = pd.DataFrame([{
    "Model": "Random Forest",
    "Best-F1 Score": best_f1,
    "Accuracy": accuracy,
    "Macro-Avg F1": macro_f1,
    "Weighted-Avg F1": weighted_f1,
    "ROC AUC": roc_auc,
    "PR AUC": pr_auc
}])

print("\nSummary table:")
print(rf_result.round(4))

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    accuracy_score,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report
)

#################################################################
#                Model 2: XGBoost
#################################################################
print("\n" + "="*20 + " Model 2: XGBoost " + "="*20 + "\n")

selector = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=42),
    max_features=200
)

X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)

param_dist_xgb = {
    'n_estimators': [200, 400, 600],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8],
    'gamma': [0.1, 0.5, 1],
    'min_child_weight': [1, 5, 10],
    'scale_pos_weight': [
        len(y_train[y_train == 0]) / len(y_train[y_train == 1]) * 1.2
    ]
}

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

random_search_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=50,
    scoring='roc_auc',
    cv=5,
    verbose=0,
    n_jobs=-1,
    random_state=42
)

random_search_xgb.fit(X_train_sel, y_train)
best_xgb_model = random_search_xgb.best_estimator_

print(f"Best Params (XGBoost): {random_search_xgb.best_params_}")

y_prob_xgb = best_xgb_model.predict_proba(X_test_sel)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_xgb)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-6)

best_idx = np.argmax(f1_scores[:-1])
best_thresh_xgb = thresholds[best_idx]

y_pred_xgb = (y_prob_xgb >= best_thresh_xgb).astype(int)

best_f1_xgb = f1_scores[best_idx]
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
macro_f1_xgb = f1_score(y_test, y_pred_xgb, average='macro')
weighted_f1_xgb = f1_score(y_test, y_pred_xgb, average='weighted')
roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb)
pr_auc_xgb = average_precision_score(y_test, y_prob_xgb)

print(f"Optimal threshold (XGBoost): {best_thresh_xgb:.4f}")
print(f"Best-F1 Score (XGBoost): {best_f1_xgb:.4f}")
print(f"Accuracy (XGBoost): {accuracy_xgb:.4f}")
print(f"Macro-Avg F1 (XGBoost): {macro_f1_xgb:.4f}")
print(f"Weighted-Avg F1 (XGBoost): {weighted_f1_xgb:.4f}")
print(f"ROC AUC (XGBoost): {roc_auc_xgb:.4f}")
print(f"PR AUC (XGBoost): {pr_auc_xgb:.4f}")

print("\nClassification report (XGBoost):")
print(classification_report(y_test, y_pred_xgb))

xgb_result = pd.DataFrame([{
    "Model": "XGBoost",
    "Best-F1 Score": best_f1_xgb,
    "Accuracy": accuracy_xgb,
    "Macro-Avg F1": macro_f1_xgb,
    "Weighted-Avg F1": weighted_f1_xgb,
    "ROC AUC": roc_auc_xgb,
    "PR AUC": pr_auc_xgb
}])

print("\nSummary table:")
print(xgb_result.round(4))

In [ ]:
#################################################################
#                   Model 3: LightGBM (LGBM)
#################################################################
print("\n" + "="*20 + " Model 3: LightGBM (LGBM) " + "="*20 + "\n")

from sklearn.metrics import (
    accuracy_score,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report
)

param_dist_lgbm = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [4, 6, 8, 10, -1],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [20, 31, 40, 50],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 0.5],
    'scale_pos_weight': [
        sum(y_train_res == 0) / sum(y_train_res == 1) * f
        for f in [0.8, 1, 1.2]
    ]
}

lgbm = lgb.LGBMClassifier(
    objective='binary',
    random_state=42,
    verbosity=-1
)

random_search_lgbm = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist_lgbm,
    n_iter=50,
    scoring='f1',
    cv=3,
    verbose=0,
    n_jobs=-1,
    random_state=42
)

random_search_lgbm.fit(X_train_pca, y_train_res)

print("Optimal parameters (LightGBM):", random_search_lgbm.best_params_)

best_lgbm_model = random_search_lgbm.best_estimator_

y_prob_lgbm = best_lgbm_model.predict_proba(X_test_pca)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_lgbm)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-6)

best_idx = np.argmax(f1_scores[:-1])
best_thresh_lgbm = thresholds[best_idx]

y_pred_lgbm = (y_prob_lgbm >= best_thresh_lgbm).astype(int)

best_f1_lgbm = f1_scores[best_idx]
accuracy_lgbm = accuracy_score(y_test, y_pred_lgbm)
macro_f1_lgbm = f1_score(y_test, y_pred_lgbm, average='macro')
weighted_f1_lgbm = f1_score(y_test, y_pred_lgbm, average='weighted')
roc_auc_lgbm = roc_auc_score(y_test, y_prob_lgbm)
pr_auc_lgbm = average_precision_score(y_test, y_prob_lgbm)

print(f"Optimal threshold (LightGBM): {best_thresh_lgbm:.4f}")
print(f"Best-F1 Score (LightGBM): {best_f1_lgbm:.4f}")
print(f"Accuracy (LightGBM): {accuracy_lgbm:.4f}")
print(f"Macro-Avg F1 (LightGBM): {macro_f1_lgbm:.4f}")
print(f"Weighted-Avg F1 (LightGBM): {weighted_f1_lgbm:.4f}")
print(f"ROC AUC (LightGBM): {roc_auc_lgbm:.4f}")
print(f"PR AUC (LightGBM): {pr_auc_lgbm:.4f}")

print("\nClassification report (LightGBM):")
print(classification_report(y_test, y_pred_lgbm))

lgbm_result = pd.DataFrame([{
    "Model": "LightGBM",
    "Best-F1 Score": best_f1_lgbm,
    "Accuracy": accuracy_lgbm,
    "Macro-Avg F1": macro_f1_lgbm,
    "Weighted-Avg F1": weighted_f1_lgbm,
    "ROC AUC": roc_auc_lgbm,
    "PR AUC": pr_auc_lgbm
}])

print("\nSummary table:")
print(lgbm_result.round(4))

In [ ]:
#################################################################
#                      Model 4: Logistic Regression
#################################################################
print("\n" + "="*20 + " Model 4: Logistic Regression " + "="*20 + "\n")

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report
)

pca_lr = PCA(n_components=100, random_state=42)
X_train_pca_lr = pca_lr.fit_transform(X_train_res)
X_test_pca_lr = pca_lr.transform(X_test)

param_grid_enet = {
    'C': [0.1, 1, 10, 100],
    'penalty': ['elasticnet'],
    'solver': ['saga'],
    'l1_ratio': [0.1, 0.5, 0.9],
    'class_weight': ['balanced', None]
}

lr_enet = LogisticRegression(
    max_iter=10000,
    random_state=42
)

grid_enet = GridSearchCV(
    lr_enet,
    param_grid_enet,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)

grid_enet.fit(X_train_pca_lr, y_train_res)

print(f"Optimal parameters ElasticNet: {grid_enet.best_params_}")

best_enet = grid_enet.best_estimator_

probs_enet = best_enet.predict_proba(X_test_pca_lr)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, probs_enet)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-6)

best_idx = np.argmax(f1_scores[:-1])
best_thresh_lr = thresholds[best_idx]

y_pred_enet = (probs_enet >= best_thresh_lr).astype(int)

best_f1_lr = f1_scores[best_idx]
accuracy_lr = accuracy_score(y_test, y_pred_enet)
macro_f1_lr = f1_score(y_test, y_pred_enet, average='macro')
weighted_f1_lr = f1_score(y_test, y_pred_enet, average='weighted')
roc_auc_lr = roc_auc_score(y_test, probs_enet)
pr_auc_lr = average_precision_score(y_test, probs_enet)

print(f"Optimal threshold (Logistic Regression): {best_thresh_lr:.4f}")
print(f"Best-F1 Score (Logistic Regression): {best_f1_lr:.4f}")
print(f"Accuracy (Logistic Regression): {accuracy_lr:.4f}")
print(f"Macro-Avg F1 (Logistic Regression): {macro_f1_lr:.4f}")
print(f"Weighted-Avg F1 (Logistic Regression): {weighted_f1_lr:.4f}")
print(f"ROC AUC (Logistic Regression): {roc_auc_lr:.4f}")
print(f"PR AUC (Logistic Regression): {pr_auc_lr:.4f}")

print("\nClassification report (Logistic Regression):")
print(classification_report(y_test, y_pred_enet))

lr_result = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Best-F1 Score": best_f1_lr,
    "Accuracy": accuracy_lr,
    "Macro-Avg F1": macro_f1_lr,
    "Weighted-Avg F1": weighted_f1_lr,
    "ROC AUC": roc_auc_lr,
    "PR AUC": pr_auc_lr
}])

print("\nSummary table:")
print(lr_result.round(4))